# 03 · Seq2Seq + Attention —— 让解码时回头看原文

**家族位置**：`04_Sequence_Models` 第 3 站（编解码）。01 手拉手标、02 会记但只能分类；本章把“记住一句”升级为“生成一句”——Seq2Seq 先把整句压成一个向量再吐新句，Attention 让吐每个词时回头看原文哪块最相关。

**学习目标**
1. 瓶颈：定长向量压 6 个词→信息挤爆，长句必糊
2. Attention 如何救：给解码每步配“聚光灯”，按相关度加权原文，瓶颈被旁路
3. 热力图：复制任务（对角线）vs 翻转任务（反对角）——注意力学会对齐
4. 为 05 铺路：Attention 就是 Transformer 的本体，这里是 Bahdanau 加法式原型

## 1. 原理：从“背整句”到“边看边翻”

### 通俗理解

**一句话**：Seq2Seq 是“听完整句再复述”——编码器把 6 个字压成 1 个向量，解码器只看这 1 个向量往外吐；Attention 是“边翻边回头看底稿”——吐第 3 个词时，眼睛盯回原文第 3（复制）或第 4（翻转）个字。

**比喻**：无 Attention 像闭卷复述——全靠记忆，6 词尚可，30 词必忘；有 Attention 像开卷翻译——底稿铺在桌上，翻到哪看哪，聚光灯（权重）指哪打哪。

### 结构账

```
编码：  src(6) → Emb → GRU → enc_out(6×32) + h(32)          6 个隐状态全保留
无Attn： dec 只看 h 去吐 6 词                               瓶颈=1 向量
有Attn： score = vᵀ tanh(W[dec_h; enc_t]) → softmax → ctx  每步重算 6 维权重，ctx=Σα·enc
解码：  dec_h' = GRUCell([emb[tgt_t]; ctx], dec_h) → out([dec_h'; ctx])  聚光灯旁路瓶颈
```

- **复制** `src==tgt`：Attention 应学对角线（t 对齐 t）
- **翻转** `tgt=reverse(src)`：应学反对角线（t 对齐 5-t），需长程重排，瓶颈更痛

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import make_copy_data, make_reverse_data
from common.models import Seq2SeqAttention
from common.utils import set_seed, setup_chinese_font, count_params

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE, "| torch:", torch.__version__)

VOCAB = 8
SEQ = 6
N = 600
src_copy, tgt_copy = make_copy_data(n=N, seq_len=SEQ, vocab_size=VOCAB, seed=0)
src_rev, tgt_rev = make_reverse_data(n=N, seq_len=SEQ, vocab_size=VOCAB, seed=1)
print(f"复制 {len(src_copy)} ×{SEQ} vocab{VOCAB}  例 src {src_copy[0]} → tgt {tgt_copy[0]}")
print(f"翻转 {len(src_rev)} ×{SEQ} vocab{VOCAB}  例 src {src_rev[0]} → tgt {tgt_rev[0]}")


## 2. 数据：复制（对角）与翻转（反对角）—— 轻量但足以让注意力对齐现形

In [ ]:
# fig0：2 任务样例条带
fig, axes = plt.subplots(2, 1, figsize=(10, 2.8))
for ax, (src, tgt, title) in zip(axes, [(src_copy[0], tgt_copy[0], "复制 src==tgt（对角）"), (src_rev[0], tgt_rev[0], "翻转 tgt=reverse(src)（反对角）")]):
    ax.set_xlim(0, SEQ)
    ax.set_ylim(0, 1.1)
    ax.axis("off")
    ax.set_title(title, fontsize=9, loc="left")
    for i, (s, t) in enumerate(zip(src, tgt)):
        ax.add_patch(plt.Rectangle((i+0.08, 0.52), 0.84, 0.42, facecolor="#D6EAF8", edgecolor="#2E86C1", linewidth=0.9))
        ax.text(i+0.5, 0.73, str(s), ha="center", va="center", fontsize=9)
        ax.add_patch(plt.Rectangle((i+0.08, 0.06), 0.84, 0.42, facecolor="#FCF3CF", edgecolor="#B7950B", linewidth=0.9))
        ax.text(i+0.5, 0.27, str(t), ha="center", va="center", fontsize=9)
    ax.text(SEQ+0.2, 0.73, "src", va="center", fontsize=8, color="#2E86C1")
    ax.text(SEQ+0.2, 0.27, "tgt", va="center", fontsize=8, color="#7D6608")
fig.suptitle("Toy Seq2Seq：复制 vs 翻转（S=6，vocab 8）", fontsize=10)
plt.tight_layout()
plt.savefig(FIGS / "fig0_task.png", dpi=150, bbox_inches="tight")
plt.show()

m = Seq2SeqAttention(vocab_size=VOCAB, emb_dim=16, hid_dim=32)
print(f"Seq2Seq+Attention 参数 {count_params(m):5d}  emb16 hid32")


## 3. 主实验：同参同训 60ep，复制 vs 翻转 seq-acc 与注意力热力图

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

BOS = VOCAB

def train_seq2seq(src_list, tgt_list, epochs=60, batch_size=32, lr=8e-3, device=DEVICE, seed=0):
    set_seed(seed)
    model = Seq2SeqAttention(vocab_size=VOCAB, emb_dim=16, hid_dim=32).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    src = torch.tensor(src_list, dtype=torch.long)
    tgt = torch.tensor(tgt_list, dtype=torch.long)
    tgt_input = torch.cat([torch.full((tgt.size(0),1), BOS, dtype=torch.long), tgt[:,:-1]], dim=1)
    ds = TensorDataset(src, tgt, tgt_input)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    src_d = src.to(device)
    tgt_d = tgt.to(device)
    tgt_in_d = tgt_input.to(device)
    hist = []
    for ep in range(1, epochs+1):
        model.train()
        for s, t, ti in loader:
            s, t, ti = s.to(device), t.to(device), ti.to(device)
            opt.zero_grad()
            logits, _ = model(s, ti)
            loss = crit(logits.reshape(-1, VOCAB), t.reshape(-1))
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            logits, _ = model(src_d, tgt_in_d)
            pred = logits.argmax(-1).cpu().numpy()
            acc = (pred == __import__("numpy").array(tgt_list)).all(axis=1).mean()
        hist.append(float(acc))
        if ep==1 or ep%10==0 or ep==epochs:
            print(f"epoch {ep:02d}/{epochs}  seq-acc {acc:.3f}", flush=True)
    return model, hist

print("—— 复制任务 ——")
model_copy, hist_copy = train_seq2seq(src_copy, tgt_copy, epochs=60, seed=0)
print("\n—— 翻转任务 ——")
model_rev, hist_rev = train_seq2seq(src_rev, tgt_rev, epochs=60, seed=1)

fig, ax = plt.subplots(figsize=(7, 4))
xs = range(1, 61)
ax.plot(xs, hist_copy, label="复制 copy", color="#2E86C1", marker="o", ms=2)
ax.plot(xs, hist_rev, label="翻转 reverse", color="#E74C3C", marker="o", ms=2)
ax.set_xlabel("epoch"); ax.set_ylabel("seq-acc（整句全对）")
ax.set_title("Seq2Seq+Attention：复制 vs 翻转 seq-acc（S=6，N=600，同参同训 60ep）")
ax.set_ylim(0, 1.05); ax.legend()
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"终局 seq-acc：复制 {hist_copy[-1]:.3f}  翻转 {hist_rev[-1]:.3f}")


## 4. 注意力热力图：复制对角 vs 翻转反对角

In [ ]:
def plot_attention(model, src_list, title, save_name, row=0):
    model.eval()
    src = torch.tensor([src_list[row]], dtype=torch.long, device=DEVICE)
    tgt_real = torch.tensor([src_list[row] if "复制" in title else src_list[row][::-1]], dtype=torch.long, device=DEVICE)
    tgt_in = torch.cat([torch.tensor([[VOCAB]], dtype=torch.long, device=DEVICE), tgt_real[:,:-1]], dim=1)
    with torch.no_grad():
        _, attns = model(src, tgt_in)
    attn = attns[0].cpu().numpy()
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    im = ax.imshow(attn, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xlabel("src 位置")
    ax.set_ylabel("tgt 位置")
    ax.set_xticks(range(SEQ)); ax.set_yticks(range(SEQ))
    ax.set_title(title)
    for i in range(SEQ):
        for j in range(SEQ):
            ax.text(j, i, f"{attn[i,j]:.2f}", ha="center", va="center", fontsize=6, color="white" if attn[i,j]>0.5 else "black")
    plt.colorbar(im, ax=ax, shrink=0.9)
    plt.tight_layout()
    plt.savefig(FIGS / save_name, dpi=150, bbox_inches="tight")
    plt.show()
    return attn

attn_copy = plot_attention(model_copy, src_copy, "复制·Attention（应对角线）", "fig2_attn_copy.png", row=0)
attn_rev = plot_attention(model_rev, src_rev, "翻转·Attention（应反对角）", "fig3_attn_reverse.png", row=0)

model_copy.eval()
with torch.no_grad():
    for k in range(3):
        src = torch.tensor([src_copy[k]], dtype=torch.long, device=DEVICE)
        pred, _ = model_copy.greedy_decode(src, max_len=SEQ)
        pred = pred[0].cpu().tolist()
        tag = "✓" if pred==tgt_copy[k] else "✗"
        print(f"复制例{k+1}  src {src_copy[k]}  tgt {tgt_copy[k]}  pred {pred}  {tag}")
with torch.no_grad():
    for k in range(3):
        src = torch.tensor([src_rev[k]], dtype=torch.long, device=DEVICE)
        pred, _ = model_rev.greedy_decode(src, max_len=SEQ)
        pred = pred[0].cpu().tolist()
        tag = "✓" if pred==tgt_rev[k] else "✗"
        print(f"翻转例{k+1}  src {src_rev[k]}  tgt {tgt_rev[k]}  pred {pred}  {tag}")


## 5. 总结与下一步

**本项目收获**

1. Seq2Seq 瓶颈：单向量压 6 词可训，但注意力让对齐可视化——复制对角、翻转反对角即证据
2. 翻转比复制难：需长程重排，seq-acc 收敛慢 10~15ep，但 Attention 仍能学到反对角
3. Bahdanau 加法式 Attention 就是 Transformer 自注意力的雏形：score→softmax→加权和 的三步已齐
4. 同参与 02 呼应：门控（LSTM/GRU）救“纵向消失”，Attention 救“横向瓶颈”——两类直通思想殊途同归

**下一步**：`05_Transformer_NLP`——把“循环+注意力”换成“全注意力无循环”（Transformer），在同 toy 上对比并行与 O(n²) 的代价；`04-04 Mamba` 学完 05 后以线性复杂度回补对照。